# 🧑‍💻 Session 6: Retrieval Augmented Generation (RAG)

RAG combines:
1. **Retriever** → Fetches relevant documents from a vector store.  
2. **LLM** → Generates context-aware responses using query + retrieved docs.  

In this session, we will:
- Store documents with **Gemini Embeddings**  
- Retrieve docs from **Chroma**  
- Use **Groq LLM** for answering questions  


In [ ]:
# 📌 Install dependencies
!pip install -q langchain==1.0.5 langchain-classic==1.0.0 langchain-community==0.4.1 langchain-groq==1.0.0 langchain-google-genai==3.0.1 chromadb==1.3.4


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 10.7 MB/s eta 0:00:00

In [3]:
!pip show langchain langchain-classic langchain-community langchain-groq langchain-google-genai chromadb

Name: langchain
Version: 1.0.5
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-classic
Version: 1.0.0
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langchain-text-splitters, langsmith, pydantic, pyyaml, requests, sqlalchemy
Required-by: langchain-community
---
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tena

## 🔑 Setup API Keys
- Google Gemini API key → for embeddings  
- Groq API key → for LLM (LLaMA models)


In [4]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

## 📄 Step 1: Create Documents
We’ll use IPL players knowledge base.


In [5]:
from langchain_classic.schema import Document

docs = [
    Document(
        page_content="Virat Kohli is one of the most successful batsmen in IPL history and has captained RCB.",
        metadata={"team": "Royal Challengers Bangalore"}
    ),
    Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, winning five titles with Mumbai Indians.",
        metadata={"team": "Mumbai Indians"}
    ),
    Document(
        page_content="MS Dhoni has led Chennai Super Kings to multiple IPL titles and is known as Captain Cool.",
        metadata={"team": "Chennai Super Kings"}
    ),
    Document(
        page_content="Jasprit Bumrah is a leading fast bowler for Mumbai Indians, famous for his yorkers.",
        metadata={"team": "Mumbai Indians"}
    ),
    Document(
        page_content="Ravindra Jadeja is an all-rounder for Chennai Super Kings, contributing with bat, ball, and fielding.",
        metadata={"team": "Chennai Super Kings"}
    )
]


## 🗂️ Step 2: Store Documents with Gemini Embeddings in Chroma
We’ll use **GoogleGenerativeAIEmbeddings** for vector representation.


In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GEMINI_API_KEY)

# Create Chroma vector store
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="rag_chroma_db",
    collection_name="ipl_docs"
)

# Add documents
vector_store.add_documents(docs)


/tmp/ipython-input-931093050.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


['75136021-9efc-4434-bd97-a77c659b068c',
 'd424360c-86c9-4301-aa49-fd9b68c9ac0e',
 'e60b5b5c-27ae-4327-91b2-30a88054af0c',
 'df544b98-b033-4aba-b7b1-7dbe11bf0f54',
 'c025bdae-d153-4b6b-9d2b-ac107986f10c']

## 🔎 Step 3: Create Retriever
Retriever fetches relevant chunks from Chroma.


In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

## 🧠 Step 4: Initialize Groq LLM
We use Groq-hosted LLaMA 3.


In [8]:
from langchain_groq import ChatGroq
from google.colab import userdata
from langchain_groq import ChatGroq

# Load API key
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0.3,
    max_tokens=200
)



## 🔗 Step 5: Create RAG Chain
Combine retriever + LLM into a RetrievalQA pipeline.


In [11]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)


## 💬 Step 6: Ask Questions
Test RAG pipeline with cricket-related queries.


In [12]:
# Query 1
query = "Who is the most successful IPL captain?"
response = qa_chain.invoke({"query": query})

print("Query:", query)
print("Answer:", response["result"])
print("\nSources:", [doc.metadata for doc in response["source_documents"]])

# Query 2
query2 = "Which bowler is famous for yorkers?"
response2 = qa_chain.invoke({"query": query2})

print("\nQuery:", query2)
print("Answer:", response2["result"])
print("\nSources:", [doc.metadata for doc in response2["source_documents"]])


Query: Who is the most successful IPL captain?
Answer: The most successful IPL captain is **Rohit Sharma**, who has led Mumbai Indians to five titles.

Sources: [{'team': 'Mumbai Indians'}, {'team': 'Royal Challengers Bangalore'}]

Query: Which bowler is famous for yorkers?
Answer: The bowler famous for his yorkers is **Jasprit Bumrah**.

Sources: [{'team': 'Mumbai Indians'}, {'team': 'Mumbai Indians'}]


# ✅ Summary
- Used **Google Gemini Embeddings** to vectorize documents  
- Stored + Retrieved docs from **Chroma**  
- Connected retriever with **Groq LLM**  
- Answered queries using **RAG pipeline**  

👉 Next session: **Advanced RAG – Custom Prompts, Comparisons, and Deep Dive**
